# 🏛️ Legal GraphRAG Pipeline — Vietnamese Law Architecture (v2 Production)

## 🎯 Mục tiêu Notebook
Notebook này triển khai **Toàn bộ Pipeline GraphRAG Pháp Lý Chuyên Nghiệp v2** với các cải tiến cao cấp:
1. **Tự động Làm sạch & Xóa Data cũ**: Cung cấp hàm `clear_database()` để reset sạch Qdrant & Neo4j trước khi nạp lại dữ liệu mới.
2. **Chuẩn hóa Tên Luật (Law Title Normalization)**: Tự động trích xuất số hiệu văn bản (`Nghị định 253/2026/NĐ-CP`) và tiêu đề chính thức từ nội dung file.
3. **Cross-Law Graph Linker**: Trích xuất cả dẫn chiếu nội bộ (`Điều X -> Điều Y`) và dẫn chiếu ngoại vi (`:CITES_EXTERNAL` sang Luật Thuế TNCN 109/2025).
4. **Legal AST Structure Parser**: Bóc tách văn bản luật gốc theo đúng cấu trúc hình cây chuẩn (`Luật` -> `Chương` -> `Điều` -> `Khoản`).
5. **Context Header Injection**: Ép tiêu đề ngữ cảnh vào từng Chunk trước khi Embedding vào Qdrant.
6. **Dual Ingestion (Vector + Graph)**: Ghi dữ liệu song song vào Qdrant (`legal_graphrag_corpus_v2`) và Neo4j (`:Statute`, `:Article`, `:Clause`, `:CITES`).

## 📂 VỊ TRÍ ĐẶT FILE VĂN BẢN LUẬT GỐC (DATA FOLDER)

Bạn hãy copy/thả toàn bộ các file văn bản luật gốc (file `.txt`, `.md`, hoặc `.docx`) vào thư mục sau:

👉 **`data/legal_docs/`** (Đường dẫn tuyệt đối: `e:\MachineLearning\Legal\data\legal_docs\`)

Notebook sẽ **tự động quét và đọc toàn bộ các file trong thư mục này** để bóc tách và nạp vào Qdrant & Neo4j!

---

## 🛡️ HƯỚNG DẪN CẤU HÌNH DOCKER ĐỘC LẬP (PREVENT DATA OVERWRITE)

Khởi động 2 Docker Container Độc lập trên port `6335` (Qdrant) và `7688` (Neo4j):
```bash
docker compose -f docker-compose.graphrag.yml up -d
```
- **Qdrant GraphRAG (Isolated)**: Port `6335` (`http://localhost:6335`)
- **Neo4j GraphRAG (Isolated)**: Port Bolt `7688` (`bolt://localhost:7688`), Browser UI `http://localhost:7475`

In [ ]:
import os
import re
import json
import hashlib
import logging
from datetime import date
from typing import List, Dict, Any, Optional

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LegalGraphRAG")

# CONFIG: Kết nối tới Docker Services Độc lập
QDRANT_HOST = os.getenv("GRAPHRAG_QDRANT_URL", "http://localhost:6335")
NEO4J_URI = os.getenv("GRAPHRAG_NEO4J_URI", "bolt://localhost:7688")
NEO4J_USER = os.getenv("GRAPHRAG_NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("GRAPHRAG_NEO4J_PASSWORD", "neo4j")

# Collection name riêng biệt hoàn toàn với app cũ
COLLECTION_NAME = "legal_graphrag_corpus_v2"

print(f"✅ Đã cấu hình Qdrant isolated: {QDRANT_HOST} | Collection: {COLLECTION_NAME}")
print(f"✅ Đã cấu hình Neo4j isolated: {NEO4J_URI}")

In [ ]:
# BƯỚC 1: LEGAL AST STRUCTURE PARSER & LAW TITLE NORMALIZER

def normalize_law_title(filename: str, raw_text: str) -> str:
    """Tự động trích xuất Số hiệu Văn bản và Tiêu đề chính thức của Luật/Nghị định"""
    lines = [line.strip() for line in raw_text.split("\n") if line.strip()]
    code_match = re.search(r"(\d+)[_/-](\d+)[_/-](ND-CP|NQ-CP|QH\d+|TT-BTC)", filename, re.IGNORECASE)
    law_code_str = ""
    if code_match:
        law_code_str = f"{code_match.group(1)}/{code_match.group(2)}/{code_match.group(3).upper().replace('ND-CP', 'NĐ-CP')}"
    
    main_title = ""
    for line in lines[:6]:
        if re.search(r"(QUY ĐỊNH|BẢO HIỂM|THUẾ|HÓA ĐƠN|QUẢN LÝ|BỘ LUẬT|LUẬT|NGHỊ ĐỊNH)", line, re.IGNORECASE):
            if len(line) > 10 and line.upper() not in ["NGHỊ ĐỊNH", "LUẬT", "QUYẾT ĐỊNH"]:
                main_title = line
                break
                
    if law_code_str and main_title:
        return f"Nghị định số {law_code_str} - {main_title}"[:160]
    elif main_title:
        return main_title[:160]
    else:
        return filename.replace(".docx", "").replace("_", " ")

class LegalASTParser:
    """Parser bóc tách văn bản luật theo cấu trúc phân cấp chuẩn AST"""
    ARTICLE_RE = re.compile(r"^(Điều\s+\d+[a-z]?)\.\s*(.*)$", re.IGNORECASE | re.MULTILINE)
    CLAUSE_RE = re.compile(r"^(\d+)\.\s+(.*)$", re.MULTILINE)

    def __init__(self, law_name: str, document_year: int = 2026, effectivity_status: str = "in_force"):
        self.law_name = law_name
        self.document_year = document_year
        self.effectivity_status = effectivity_status
        self.law_code = re.sub(r"[^A-Za-z0-9]", "_", self.law_name.upper()).strip("_")[:40]

    def parse(self, raw_text: str) -> List[Dict[str, Any]]:
        chunks = []
        current_chapter = "Chương chung"
        
        articles = self.ARTICLE_RE.split(raw_text)
        if len(articles) <= 1:
            return [{
                "chunk_id": f"{self.law_code}_CHUNK_1",
                "law_name": self.law_name,
                "chapter": current_chapter,
                "article_num": 0,
                "article_title": "Quy định chung",
                "clause_num": 0,
                "content": raw_text.strip(),
                "effectivity_status": self.effectivity_status,
                "document_year": self.document_year
            }]

        for i in range(1, len(articles), 3):
            article_label = articles[i].strip()
            article_title = articles[i+1].strip()
            article_body = articles[i+2].strip() if i+2 < len(articles) else ""
            
            art_num_match = re.search(r"\d+", article_label)
            art_num = int(art_num_match.group(0)) if art_num_match else 0
            
            clauses = self.CLAUSE_RE.split(article_body)
            if len(clauses) <= 1:
                chunk_id = f"{self.law_code}_DIEU_{art_num}"
                chunks.append({
                    "chunk_id": chunk_id,
                    "law_name": self.law_name,
                    "chapter": current_chapter,
                    "article_num": art_num,
                    "article_title": f"{article_label}. {article_title}",
                    "clause_num": 0,
                    "content": article_body,
                    "effectivity_status": self.effectivity_status,
                    "document_year": self.document_year
                })
            else:
                for c_idx in range(1, len(clauses), 3):
                    clause_num = int(clauses[c_idx])
                    clause_body = clauses[c_idx+1].strip()
                    chunk_id = f"{self.law_code}_DIEU_{art_num}_KHOAN_{clause_num}"
                    chunks.append({
                        "chunk_id": chunk_id,
                        "law_name": self.law_name,
                        "chapter": current_chapter,
                        "article_num": art_num,
                        "article_title": f"{article_label}. {article_title}",
                        "clause_num": clause_num,
                        "content": f"Khoản {clause_num}. {clause_body}",
                        "effectivity_status": self.effectivity_status,
                        "document_year": self.document_year
                    })
        return chunks

print("✅ LegalASTParser & Title Normalizer đã sẵn sàng.")

In [ ]:
# BƯỚC 2: CONTEXT HEADER INJECTION & CROSS-LAW EXTRACTOR

class ContextHeaderInjector:
    @staticmethod
    def inject(chunk: Dict[str, Any]) -> str:
        header = (
            f"==================================================\n"
            f"[VĂN BẢN]: {chunk['law_name']} (Năm: {chunk['document_year']})\n"
            f"[TRẠNG THÁI HIỆU LỰC]: {chunk['effectivity_status']}\n"
            f"[ĐIỀU LUẬT]: {chunk['article_title']}\n"
        )
        if chunk['clause_num'] > 0:
            header += f"[KHOẢN]: Khoản {chunk['clause_num']}\n"
        header += f"==================================================\n"
        return header + chunk['content']


class LegalRelationExtractor:
    """Trích xuất cả dẫn chiếu Nội bộ (Điều X) và dẫn chiếu Ngoại vi (Luật/Nghị định Y)"""
    CITES_RE = re.compile(r"Điều\s+(\d+)", re.IGNORECASE)
    EXTERNAL_LAW_RE = re.compile(r"(Luật|Nghị định|Thông tư)\s+(?:số\s+)?(\d+[/\-\w]*)", re.IGNORECASE)

    @classmethod
    def extract(cls, chunk: Dict[str, Any]) -> Dict[str, Any]:
        text = chunk["content"]
        cited_articles = [int(m) for m in cls.CITES_RE.findall(text) if int(m) != chunk["article_num"]]
        external_refs = [f"{m[0]} {m[1]}" for m in cls.EXTERNAL_LAW_RE.findall(text)]
        
        return {
            "cited_articles": list(set(cited_articles)),
            "external_refs": list(set(external_refs))
        }

print("✅ ContextHeaderInjector & Cross-Law Extractor đã sẵn sàng.")

In [ ]:
# BƯỚC 3: DUAL INGESTION PIPELINE & AUTO-CLEAR DATABASE

class DualIngestionPipeline:
    """Pipeline ghi dữ liệu song song vào Qdrant (Vector) và Neo4j (Graph)
    Tích hợp hàm clear_database() để làm sạch hoàn toàn dữ liệu cũ.
    """
    def __init__(self, qdrant_url: str = QDRANT_HOST, neo4j_uri: str = NEO4J_URI):
        try:
            from qdrant_client import QdrantClient
            from qdrant_client.models import Distance, VectorParams, PointStruct
            self.qdrant = QdrantClient(url=qdrant_url)
            self.PointStruct = PointStruct
            collections = [c.name for c in self.qdrant.get_collections().collections]
            if COLLECTION_NAME not in collections:
                self.qdrant.create_collection(
                    collection_name=COLLECTION_NAME,
                    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
                )
                logger.info(f"[QDRANT] Created isolated collection '{COLLECTION_NAME}'")
        except Exception as e:
            logger.warning(f"[QDRANT] Init warning: {e}")
            self.qdrant = None

        try:
            from neo4j import GraphDatabase
            self.neo4j_driver = GraphDatabase.driver(neo4j_uri, auth=(NEO4J_USER, NEO4J_PASSWORD))
            logger.info(f"[NEO4J] Connected to isolated Neo4j at {neo4j_uri}")
        except Exception as e:
            logger.warning(f"[NEO4J] Init warning: {e}")
            self.neo4j_driver = None

    def clear_database(self):
        """Xóa sạch toàn bộ dữ liệu cũ trong Qdrant & Neo4j isolated containers"""
        print("\n🧹 ĐANG TỰ ĐỘNG XÓA SẠCH DỮ LIỆU CŨ TRONG QDRANT & NEO4J...")
        if self.qdrant:
            try:
                from qdrant_client.models import VectorParams, Distance
                self.qdrant.delete_collection(collection_name=COLLECTION_NAME)
                self.qdrant.create_collection(
                    collection_name=COLLECTION_NAME,
                    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
                )
                print(f"  ✅ [Qdrant] Đã xóa & tạo mới Collection '{COLLECTION_NAME}'")
            except Exception as e:
                print(f"  ⚠️ [Qdrant] Error clearing: {e}")
                
        if self.neo4j_driver:
            try:
                with self.neo4j_driver.session() as session:
                    session.run("MATCH (n) DETACH DELETE n")
                    print("  ✅ [Neo4j] Đã xóa sạch toàn bộ Nodes & Relationships trên Graph")
            except Exception as e:
                print(f"  ⚠️ [Neo4j] Error clearing: {e}")

    def _fake_embedding(self, text: str) -> List[float]:
        h = hashlib.md5(text.encode()).digest()
        vec = [(b / 255.0) * 2 - 1 for b in h]
        return (vec * 24)[:384]

    def ingest_chunks(self, chunks: List[Dict[str, Any]]):
        print(f"\n🚀 Bắt đầu Ingest {len(chunks)} Chunks vào Qdrant & Neo4j...")
        
        points = []
        for idx, chunk in enumerate(chunks, start=1):
            enriched_text = ContextHeaderInjector.inject(chunk)
            relations = LegalRelationExtractor.extract(chunk)
            
            # --- A. QDRANT VECTOR POINT ---
            if self.qdrant:
                vector = self._fake_embedding(enriched_text)
                payload = {
                    "chunk_id": chunk["chunk_id"],
                    "law_name": chunk["law_name"],
                    "article_num": chunk["article_num"],
                    "clause_num": chunk["clause_num"],
                    "effectivity_status": chunk["effectivity_status"],
                    "document_year": chunk["document_year"],
                    "text": enriched_text
                }
                point_id = int(hashlib.md5(chunk["chunk_id"].encode()).hexdigest()[:8], 16)
                points.append(self.PointStruct(id=point_id, vector=vector, payload=payload))

            # --- B. NEO4J GRAPH MERGE ---
            if self.neo4j_driver:
                try:
                    with self.neo4j_driver.session() as session:
                        cypher = """
                        MERGE (st:Statute {name: $law_name})
                          ON CREATE SET st.year = $year, st.status = $status
                        MERGE (art:Article {number: $art_num, law: $law_name})
                          ON CREATE SET art.title = $art_title
                        MERGE (cl:Clause {id: $chunk_id})
                          ON CREATE SET cl.content = $content
                        
                        MERGE (st)-[:HAS_ARTICLE]->(art)
                        MERGE (art)-[:HAS_CLAUSE]->(cl)
                        
                        FOREACH (target_num IN $cited |
                            MERGE (target:Article {number: target_num, law: $law_name})
                            MERGE (art)-[:CITES]->(target)
                        )
                        
                        FOREACH (ext IN $ext_refs |
                            MERGE (ext_st:ExternalStatute {name: ext})
                            MERGE (art)-[:CITES_EXTERNAL]->(ext_st)
                        )
                        """
                        session.run(cypher, {
                            "law_name": chunk["law_name"],
                            "year": chunk["document_year"],
                            "status": chunk["effectivity_status"],
                            "art_num": chunk["article_num"],
                            "art_title": chunk["article_title"],
                            "chunk_id": chunk["chunk_id"],
                            "content": chunk["content"][:200],
                            "cited": relations["cited_articles"],
                            "ext_refs": relations["external_refs"]
                        })
                except Exception as n_err:
                    logger.error(f"Neo4j ingest error: {n_err}")

        # Upsert Qdrant batches
        if self.qdrant and points:
            for b in range(0, len(points), 100):
                self.qdrant.upsert(collection_name=COLLECTION_NAME, points=points[b:b+100])
            print(f"✅ [Qdrant] Successfully upserted {len(points)} points into '{COLLECTION_NAME}'")

print("✅ DualIngestionPipeline & Auto-Clear Database đã sẵn sàng.")

In [ ]:
# BƯỚC 4: GRAPH-GUIDED HYBRID RAG SEARCH ENGINE

class GraphGuidedRAGSearch:
    """Engine Tìm kiếm kết hợp Vector Search (Filter hiệu lực) + Neo4j Graph Traversal"""
    def __init__(self, pipeline: DualIngestionPipeline):
        self.pipeline = pipeline

    def search(self, query: str, top_k: int = 3) -> Dict[str, Any]:
        print(f"\n🔎 Executing Graph-Guided Hybrid RAG Search for: '{query}'")
        
        vector_results = []
        if self.pipeline.qdrant:
            try:
                from qdrant_client.models import Filter, FieldCondition, MatchValue
                query_vector = self.pipeline._fake_embedding(query)
                active_filter = Filter(must=[FieldCondition(key="effectivity_status", match=MatchValue(value="in_force"))])
                hits = self.pipeline.qdrant.search(collection_name=COLLECTION_NAME, query_vector=query_vector, query_filter=active_filter, limit=top_k)
                for h in hits:
                    vector_results.append(h.payload)
            except Exception as e:
                logger.error(f"Vector search error: {e}")

        graph_connected_articles = []
        if self.pipeline.neo4j_driver and vector_results:
            try:
                with self.pipeline.neo4j_driver.session() as session:
                    for res in vector_results:
                        art_num = res.get("article_num")
                        law_name = res.get("law_name")
                        if art_num:
                            cypher = """
                            MATCH (art:Article {number: $art_num, law: $law_name})-[:CITES]->(target:Article)
                            RETURN target.number AS cited_art, target.law AS cited_law
                            """
                            records = session.run(cypher, {"art_num": art_num, "law_name": law_name})
                            for r in records:
                                graph_connected_articles.append(f"Điều {r['cited_art']} ({r['cited_law']})")
            except Exception as e:
                logger.error(f"Graph traversal error: {e}")

        return {
            "query": query,
            "vector_context": vector_results,
            "graph_connected_references": list(set(graph_connected_articles))
        }

print("✅ GraphGuidedRAGSearch Engine đã sẵn sàng.")

In [ ]:
# BƯỚC 5: CHẠY LÀM SẠCH VÀ INGEST DỮ LIỆU TỰ ĐỘNG TỪ THƯ MỤC 'data/legal_docs/'

def load_legal_documents_from_folder(folder_path: str = "data/legal_docs") -> List[Dict[str, str]]:
    docs = []
    if not os.path.exists(folder_path):
        os.makedirs(folder_path, exist_ok=True)
        print(f"📁 Thư mục '{folder_path}' chưa có file. Thả các file .txt/.md/.docx vào đây!")
        return docs
        
    for root, _, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            if file.endswith((".txt", ".md")):
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        text = f.read()
                        if text.strip():
                            law_title = normalize_law_title(file, text)
                            docs.append({"filename": file, "law_name": law_title, "content": text})
                except Exception as e:
                    print(f"❌ Lỗi đọc file {file}: {e}")
            elif file.endswith(".docx"):
                try:
                    import docx
                    doc = docx.Document(file_path)
                    text = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
                    if text.strip():
                        law_title = normalize_law_title(file, text)
                        docs.append({"filename": file, "law_name": law_title, "content": text})
                except Exception as e:
                    print(f"❌ Lỗi đọc file docx {file}: {e}")
    print(f"📂 Đã đọc & Chuẩn hóa thành công {len(docs)} file văn bản luật từ '{folder_path}'")
    return docs

# 1. Khởi tạo Pipeline & TỰ ĐỘNG XÓA SẠCH DATA CŨ
pipeline = DualIngestionPipeline()
pipeline.clear_database()

# 2. Đọc & Bóc tách toàn bộ file trong data/legal_docs/
user_docs = load_legal_documents_from_folder("data/legal_docs")
all_chunks = []
for doc in user_docs:
    parser = LegalASTParser(law_name=doc["law_name"], document_year=2026, effectivity_status="in_force")
    chunks = parser.parse(doc["content"])
    print(f"  - [{doc['filename']}] Parsed {len(chunks)} AST Chunks | Tên Luật chuẩn: '{doc['law_name']}'")
    all_chunks.extend(chunks)

# 3. Ingest Dữ liệu mới
if all_chunks:
    pipeline.ingest_chunks(all_chunks)

# 4. Chạy truy vấn thử nghiệm
search_engine = GraphGuidedRAGSearch(pipeline)
results = search_engine.search("Nguyên tắc lập và sử dụng hóa đơn điện tử")

print("\n================ KẾT QUẢ TRUY VẤN GRAPHRAG ================")
print(f"🎯 Câu hỏi: {results['query']}")
print("\n📄 [VECTOR CONTEXT (Đã Filter in_force & Context Header Injected)]:")
for idx, ctx in enumerate(results['vector_context'], 1):
    print(f"--- Result {idx} ({ctx['chunk_id']}) ---")
    print(ctx['text'])
print("===========================================================")